# WaveForge — Run All Examples on GPU

Runs all 10 simulation examples on Colab T4 GPU, collects throughput,
generates a summary report, and commits results + plots back to GitHub.

> **Setup:** Runtime → Change runtime type → **T4 GPU** → Run all cells

**What this does:**
1. Clones the repo and installs dependencies
2. Runs all 10 examples and measures GPU throughput
3. Collects all output PNG plots
4. Generates a master comparison chart
5. Commits everything back to GitHub automatically


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode==0 else 'NOT DETECTED — enable GPU first!')


In [ ]:
!git clone https://github.com/shahzaibshazoo/waveforge.git
!pip install torch numpy matplotlib pytest --quiet
import sys, os
sys.path.insert(0, '/content/waveforge/src')
os.chdir('/content/waveforge')
print('Ready.')


In [ ]:
import re, torch, time, json, subprocess, datetime
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'
print(f'Device: {DEVICE} | GPU: {GPU_NAME}')
if DEVICE == 'cuda':
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Run All 10 Examples

In [ ]:
# Grid sizes and step counts for each example (for Mcells/s fallback calculation)
EXAMPLES = [
    ('01_basic_wave',       'examples/basic_2d_wave.py',           '128x128 free-space pulse',   128, 128,  500),
    ('02_dielectric_slab',  'examples/02_dielectric_slab.py',      '200x64  glass slab R/T',     200,  64,  800),
    ('03_waveguide',        'examples/03_waveguide.py',            '200x60  guided mode 5GHz',   200,  60, 1000),
    ('04_scattering',       'examples/04_scattering_cylinder.py',  '150x150 cylinder RCS',       150, 150,  600),
    ('05_through_wall',     'examples/05_through_wall_radar.py',   '200x100 through-wall radar', 200, 100,  600),
    ('06_interference',     'examples/06_multipath_interference.py','128x128 WiFi 2.4GHz',       128, 128,  800),
    ('07_tissue',           'examples/07_tissue_penetration.py',   '250x80  bio-tissue layers',  250,  80,  800),
    ('08_ev_radar',         'examples/08_ev_radar_ula.py',         '200x200 EV ULA 10GHz',       200, 200,  600),
    ('09_brain_clot',       'examples/09_brain_clot_dataset_sample.py','150x150 brain MIMO x16', 150, 150, 12800),
    ('10_breast_tumor',     'examples/10_breast_tumor_mimo.py',    '200x200 breast MIMO x16',    200, 200, 16000),
]

results = []
PYTHON = sys.executable

for name, script, desc, NX, NY, total_steps in EXAMPLES:
    print(f'\n[{name}] {desc}...')
    t0 = time.perf_counter()
    r  = subprocess.run([PYTHON, script], capture_output=True, text=True, timeout=900)
    elapsed = time.perf_counter() - t0

    # 1st priority: WAVEFORGE_BENCH line = sim-only timing (most accurate)
    bench = re.findall(r'WAVEFORGE_BENCH:\s*([\d]+\.?[\d]*)', r.stdout)
    if bench:
        mcells = max(float(v) for v in bench)
    else:
        # 2nd priority: any printed Mcells/s value (sim.run verbose output)
        mcells = 0.0
        for m in re.findall(r'([\d]+\.?[\d]*)\s*Mcells/s', r.stdout):
            try: mcells = max(mcells, float(m))
            except: pass
        # 3rd fallback: wall-clock / (steps * cells) — includes plot overhead, less accurate
        if mcells == 0.0 and elapsed > 0:
            mcells = round(total_steps * NX * NY / elapsed / 1e6, 1)
            print(f'  (fallback timing: {total_steps} steps, {NX}x{NY})')

    status = 'OK' if r.returncode == 0 else 'FAIL'
    results.append({
        'name':    name,
        'desc':    desc,
        'status':  status,
        'time_s':  round(elapsed, 1),
        'mcells_s': round(mcells, 1),
        'grid':    f'{NX}x{NY}',
        'steps':   total_steps,
    })
    print(f'  {status} | {elapsed:.1f}s | {mcells:.1f} Mcells/s')
    if r.returncode != 0:
        print('  STDERR:', r.stderr[-400:])

print('\nAll examples complete.')

## Results Summary

In [ ]:
print('='*68)
print(f'  WaveForge — All Examples on {GPU_NAME}')
print(f'  Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('='*68)
print(f'  {"Example":<25} {"Status":>6} {"Time":>8} {"Mcells/s":>10}')
print('-'*68)
ok = sum(1 for r in results if r['status']=='OK')
for r in results:
    star = '★' if r['mcells_s'] > 50 else ''
    print(f'  {r["name"]:<25} {r["status"]:>6} {r["time_s"]:>7.1f}s {r["mcells_s"]:>9.1f} {star}')
print('='*68)
print(f'  Passed: {ok}/{len(results)} | Total time: {sum(r["time_s"] for r in results):.0f}s')
if results:
    best = max(results, key=lambda x: x['mcells_s'])
    print(f'  Peak throughput: {best["mcells_s"]:.1f} Mcells/s ({best["name"]})')
print('='*68)


## Master Results Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'WaveForge — All Examples GPU Benchmark ({GPU_NAME})',
             fontsize=13, fontweight='bold')

names     = [r['name'].replace('_',' ') for r in results]
mcells    = [r['mcells_s'] for r in results]
times     = [r['time_s']   for r in results]
colors    = ['#2ca02c' if r['status']=='OK' else '#d62728' for r in results]

# Left: Throughput per example
ax = axes[0]
bars = ax.barh(names, mcells, color=colors, alpha=0.85)
ax.bar_label(bars, [f'{v:.0f}' for v in mcells], fontsize=9, padding=3)
ax.set(xlabel='Throughput (Mcells/s)', title='GPU Throughput per Example')
ax.axvline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3, axis='x')

# Right: Runtime per example
ax2 = axes[1]
bars2 = ax2.barh(names, times, color='steelblue', alpha=0.85)
ax2.bar_label(bars2, [f'{v:.0f}s' for v in times], fontsize=9, padding=3)
ax2.set(xlabel='Wall-clock time (s)', title='Runtime per Example')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
chart_path = '/content/waveforge/assets/all_examples_gpu_benchmark.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
# Also copy to docs/assets for website
import shutil
shutil.copy(chart_path, '/content/waveforge/docs/assets/all_examples_gpu_benchmark.png')
plt.show()
print(f'Chart saved: {chart_path}')


## Collect All Output Plots

In [ ]:
from IPython.display import Image, display
import glob, shutil

output_pngs = sorted(glob.glob('/content/waveforge/examples/output/*.png'))
print(f'Found {len(output_pngs)} output plots:')
for p in output_pngs:
    fname = Path(p).name
    dest  = f'/content/waveforge/docs/assets/simulations/{fname}'
    shutil.copy(p, dest)
    print(f'  Copied: {fname}')

# Display them all
for p in output_pngs:
    print(f'\n--- {Path(p).stem} ---')
    display(Image(p))


## Save Results JSON

In [ ]:
results_data = {
    'meta': {
        'date':     datetime.datetime.now().isoformat(),
        'gpu':      GPU_NAME,
        'platform': 'Google Colab',
        'torch':    torch.__version__,
        'device':   DEVICE,
    },
    'summary': {
        'total':   len(results),
        'passed':  sum(1 for r in results if r['status']=='OK'),
        'failed':  sum(1 for r in results if r['status']=='FAIL'),
        'total_time_s': sum(r['time_s'] for r in results),
        'peak_mcells_s': max((r['mcells_s'] for r in results), default=0),
    },
    'examples': results,
}

json_path = '/content/waveforge/benchmarks/all_examples_gpu_results.json'
with open(json_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Results saved: {json_path}')
print(json.dumps(results_data['summary'], indent=2))


## Commit Everything to GitHub

In [ ]:
# Add your GitHub token via Colab Secrets (left sidebar 🔑 key icon)
# Name: GITHUB_TOKEN | Value: your Personal Access Token (Contents: Read+Write)

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print('Token loaded from Colab Secrets')
except:
    GITHUB_TOKEN = ''  # paste here if not using Secrets
    print('Add GITHUB_TOKEN to Colab Secrets (🔑 left sidebar)')

import os, subprocess as sp
os.chdir('/content/waveforge')
sp.run(['git','config','user.name','Shahzaib Ur Rehman'])
sp.run(['git','config','user.email','shahzaib@waveforge.io'])

if GITHUB_TOKEN:
    sp.run(['git','remote','set-url','origin',
            f'https://{GITHUB_TOKEN}@github.com/shahzaibshazoo/waveforge.git'])

    # Stage everything
    sp.run(['git','add',
            'benchmarks/all_examples_gpu_results.json',
            'assets/all_examples_gpu_benchmark.png',
            'docs/assets/all_examples_gpu_benchmark.png',
            'docs/assets/simulations/'])

    msg = f'GPU benchmark: all examples on {GPU_NAME} — {datetime.datetime.now().strftime("%Y-%m-%d")}'
    sp.run(['git','commit','-m', msg])
    r = sp.run(['git','push','origin','main'], capture_output=True, text=True)
    if r.returncode == 0:
        print('Pushed to GitHub successfully!')
        print('Results at: https://github.com/shahzaibshazoo/waveforge/tree/main/benchmarks')
        print('Website:    https://shahzaibshazoo.github.io/waveforge')
    else:
        print('Push failed:', r.stderr[-500:])
else:
    print('No token — files are ready but not pushed.')
    print('Add GITHUB_TOKEN to Colab Secrets and re-run this cell.')


## Final Summary

In [ ]:
from IPython.display import Image, display
print(f'WaveForge — {len(results)} Examples on {GPU_NAME}')
print(f'Passed: {ok}/{len(results)}')
display(Image('/content/waveforge/assets/all_examples_gpu_benchmark.png'))
